# Fine-tune Qwen 3 8B — Dataset & Column Description Generation

Multi-task **QLoRA SFT followed by a DPO** preference pass, training a single adapter on two tasks:

- **`dataset_description`** — given a dataset's name + schema (column names, types, sample values), write the dataset description.
- **`column_description`** — given the dataset context + one column's name/type/samples, write that column's description.

Training data is built from `all_metadata.json` (produced by `build_metadata_store.ipynb`). The split is **held out by dataset** and saved to `splits.json`; the evaluation notebook reads the same split.

**Where this runs:** the model load + training cells need a GPU (your Tillicum H200). The data-prep cells (tagged `# @dryrun`) are pure-Python and run anywhere.

> Developed against: `transformers>=4.51` (Qwen3 support), `trl>=0.12`, `peft>=0.13`, `bitsandbytes>=0.43`, `datasets>=2.20`, `accelerate>=0.34`. A couple of TRL/Transformers kwargs have been renamed across versions — those spots are flagged inline.

## 0. Install (GPU host only)

In [ ]:
%pip install -q --upgrade "transformers>=4.51" "trl>=0.12" "peft>=0.13" "bitsandbytes>=0.45" "datasets>=2.20" "accelerate>=0.34"

## 1. Configuration

In [ ]:
# @dryrun
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-8B"  # base model to fine-tune

# --- data / artifact paths ---
METADATA_PATH = Path("all_metadata.json")
SPLITS_PATH = Path("splits.json")
DATA_DIR = Path("sft_data")
DATA_DIR.mkdir(exist_ok=True)
SFT_DIR = Path("adapters/qwen3-8b-desc-sft")
DPO_DIR = Path("adapters/qwen3-8b-desc-dpo")  # final adapter the eval notebook loads

# --- split ---
SEED = 42
VAL_FRAC, TEST_FRAC = 0.10, 0.10

# --- LoRA ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# --- sequence / batch ---
MAX_SEQ_LEN = 4096
BATCH_SIZE, GRAD_ACCUM = 2, 8  # effective batch = 16

# --- SFT ---
SFT_EPOCHS, SFT_LR = 3, 2e-4

# --- DPO ---
DPO_EPOCHS, DPO_LR, DPO_BETA = 1, 5e-6, 0.1
DPO_MAX_PAIRS = None  # None = use all constructed pairs

print("Config loaded. Final adapter ->", DPO_DIR)

In [ ]:
# --- Colab setup: locate input data + persist outputs (edit PROJECT_DIR as needed) ---
# Colab's filesystem starts empty, so all_metadata.json must be mounted from Drive or
# uploaded. This cell re-roots every artifact path under PROJECT_DIR. No-op off Colab.
from pathlib import Path

try:
    import google.colab  # are we on Colab?

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB  # set False to use Colab's local /content disk instead of Drive
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path(
        "/content/drive/MyDrive/MetadataLearn"
    )  # <-- your Drive project folder
else:
    PROJECT_DIR = Path(".")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# re-root the paths defined in the Configuration cell (so outputs persist on Drive)
METADATA_PATH = PROJECT_DIR / "all_metadata.json"
SPLITS_PATH = PROJECT_DIR / "splits.json"
DATA_DIR = PROJECT_DIR / "sft_data"
DATA_DIR.mkdir(exist_ok=True)
SFT_DIR = PROJECT_DIR / "adapters" / "qwen3-8b-desc-sft"
DPO_DIR = PROJECT_DIR / "adapters" / "qwen3-8b-desc-dpo"

# if the metadata file still isn't there, upload it once (pick all_metadata.json from your computer)
if not METADATA_PATH.exists() and IN_COLAB:
    print(
        "all_metadata.json not found at", METADATA_PATH, "- pick it from your computer:"
    )
    from google.colab import files

    up = files.upload()
    if "all_metadata.json" in up:
        METADATA_PATH.write_bytes(up["all_metadata.json"])

assert METADATA_PATH.exists(), f"Put all_metadata.json at {METADATA_PATH}"
print("Input:", METADATA_PATH, "| outputs ->", PROJECT_DIR)

## 2. Task definitions (shared with the evaluation notebook)

Keep this cell identical in both notebooks.

In [ ]:
# @dryrun  — shared task definitions (identical in finetune + evaluate notebooks)
import json, random

SYSTEM_PROMPT = (
    "You are a precise technical writer for open-data catalogs. "
    "Write grounded, concise documentation using only the provided metadata. "
    "Do not speculate or invent statistics. Output only the requested text, "
    "with no preamble, headers, or markdown."
)

MIN_DATASET_DESC_CHARS = (
    40  # skip datasets whose gold description is too thin to learn from
)
MIN_COLUMN_DESC_CHARS = 10  # skip columns with no real description
MAX_SAMPLES_DATASET = (
    4  # sample values shown per column in the dataset-description prompt
)
MAX_SAMPLES_COLUMN = 8  # sample values shown in the column-description prompt
MAX_COLS_IN_PROMPT = 40  # cap columns listed in the dataset-description prompt


def _fmt_samples(samples, n):
    vals = [str(s) for s in (samples or [])[:n] if str(s).strip()]
    return ", ".join(vals) if vals else "(no samples)"


def _column_block(columns):
    lines = []
    for c in columns[:MAX_COLS_IN_PROMPT]:
        lines.append(
            f"- {c.get('name','')} ({c.get('type','')}): e.g. "
            f"{_fmt_samples(c.get('samples'), MAX_SAMPLES_DATASET)}"
        )
    if len(columns) > MAX_COLS_IN_PROMPT:
        lines.append(f"- ... (+{len(columns) - MAX_COLS_IN_PROMPT} more columns)")
    return "\n".join(lines)


def build_dataset_desc_example(rec):
    """One dataset-description training example, or None if the dataset is unusable."""
    desc = (rec.get("description") or "").strip()
    cols = rec.get("columns") or []
    if len(desc) < MIN_DATASET_DESC_CHARS or len(cols) < 2:
        return None
    user = (
        "Task: Write a concise description (2-4 sentences) for the following dataset, "
        "suitable for an open-data catalog. Use only the provided schema and sample values.\n\n"
        f"Dataset name: {rec.get('name','')}\n"
        f"Columns:\n{_column_block(cols)}\n\n"
        "Description:"
    )
    prompt_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    return {
        "uid": rec.get("uid"),
        "task": "dataset_description",
        "column": None,
        "prompt_messages": prompt_messages,
        "messages": prompt_messages + [{"role": "assistant", "content": desc}],
        "target": desc,
    }


def build_column_desc_examples(rec):
    """All column-description training examples for one dataset."""
    out = []
    name = rec.get("name", "")
    ddesc = (rec.get("description") or "").strip()
    ddesc_short = (
        (ddesc[:300] + "...") if len(ddesc) > 300 else (ddesc or "(none provided)")
    )
    for c in rec.get("columns") or []:
        cdesc = (c.get("description") or "").strip()
        if len(cdesc) < MIN_COLUMN_DESC_CHARS:
            continue
        user = (
            "Task: Write a one-sentence description of the column named below, for the given "
            "dataset. Use only the provided context and sample values.\n\n"
            f"Dataset: {name}\n"
            f"Dataset description: {ddesc_short}\n"
            f"Column name: {c.get('name','')}\n"
            f"Column type: {c.get('type','')}\n"
            f"Sample values: {_fmt_samples(c.get('samples'), MAX_SAMPLES_COLUMN)}\n\n"
            "Column description:"
        )
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        out.append(
            {
                "uid": rec.get("uid"),
                "task": "column_description",
                "column": c.get("name"),
                "prompt_messages": prompt_messages,
                "messages": prompt_messages + [{"role": "assistant", "content": cdesc}],
                "target": cdesc,
            }
        )
    return out


def build_examples_for_uids(records, uids):
    """Return (dataset_desc_examples, column_desc_examples) for the given uids."""
    ds_ex, col_ex = [], []
    for uid in uids:
        rec = records[uid]
        d = build_dataset_desc_example(rec)
        if d:
            ds_ex.append(d)
        col_ex.extend(build_column_desc_examples(rec))
    return ds_ex, col_ex


def split_uids(uids, seed=42, val_frac=0.10, test_frac=0.10):
    """Deterministic held-out-by-dataset split so columns never leak across splits."""
    uids = sorted(uids)
    rng = random.Random(seed)
    rng.shuffle(uids)
    n = len(uids)
    n_test = max(1, round(n * test_frac))
    n_val = max(1, round(n * val_frac))
    return {
        "test": sorted(uids[:n_test]),
        "val": sorted(uids[n_test : n_test + n_val]),
        "train": sorted(uids[n_test + n_val :]),
    }

## 3. Load fetched metadata + build the split

In [ ]:
# @dryrun
records = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
all_uids = sorted(records.keys())
print(f"Loaded {len(all_uids)} datasets from {METADATA_PATH}")

splits = split_uids(all_uids, seed=SEED, val_frac=VAL_FRAC, test_frac=TEST_FRAC)
SPLITS_PATH.write_text(json.dumps(splits, indent=2), encoding="utf-8")
print({k: len(v) for k, v in splits.items()}, "-> wrote", SPLITS_PATH)

## 4. Build SFT examples (both tasks) and write them to disk

Each example is a chat conversation (`system` / `user` / `assistant`). The combined train/val sets mix both tasks; the task is signalled by the instruction text.

In [ ]:
# @dryrun
def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


sft_train, sft_val = [], []
counts = {}
for split_name, target in (("train", sft_train), ("val", sft_val)):
    ds_ex, col_ex = build_examples_for_uids(records, splits[split_name])
    target.extend(ds_ex + col_ex)
    counts[split_name] = {
        "dataset_description": len(ds_ex),
        "column_description": len(col_ex),
        "total": len(ds_ex) + len(col_ex),
    }

random.Random(SEED).shuffle(sft_train)
write_jsonl(DATA_DIR / "sft_train.jsonl", sft_train)
write_jsonl(DATA_DIR / "sft_val.jsonl", sft_val)

for k, v in counts.items():
    print(f"{k:>5}: {v}")
print(
    f"\nwrote {DATA_DIR/'sft_train.jsonl'} ({len(sft_train)}) and "
    f"{DATA_DIR/'sft_val.jsonl'} ({len(sft_val)})"
)

# peek at one example of each task
for ex in sft_train:
    if ex["task"] == "dataset_description":
        print("\n--- sample dataset_description prompt ---")
        print(ex["messages"][1]["content"][:600], "...")
        print("--- target ---\n", ex["target"][:300])
        break

## 5. Load Qwen 3 8B in 4-bit + attach LoRA  *(GPU)*

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_enable_fp32_cpu_offload=True,  # allow CPU offload for modules that don't fit in GPU
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGETS,
)
print("Base model loaded in 4-bit; LoRA targets:", LORA_TARGETS)

## 6. SFT (QLoRA)

Uses TRL's **prompt-completion** dataset format: the trainer applies the chat template and computes loss on the **completion (assistant turn) only** — automatically, with no separate data collator. (Recent TRL ≥0.20 removed `DataCollatorForCompletionOnlyLM` and renamed `max_seq_length` → `max_length`.)

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer


# Prompt-completion format: SFTTrainer applies the chat template and, for prompt/completion
# datasets, computes the loss on the completion (assistant turn) only — no separate collator
# needed. (Recent TRL removed trl.DataCollatorForCompletionOnlyLM.)
def to_prompt_completion(ex):
    return {
        "prompt": ex["prompt_messages"],
        "completion": [{"role": "assistant", "content": ex["target"]}],
    }

In [ ]:
sft_train_ds = Dataset.from_list([to_prompt_completion(e) for e in sft_train])
sft_val_ds = Dataset.from_list([to_prompt_completion(e) for e in sft_val])

sft_args = SFTConfig(
    output_dir=str(SFT_DIR),
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=SFT_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",  # older transformers: evaluation_strategy
    bf16=True,  # this GPU loaded in bf16 fine; on a T4 set bf16=False, fp16=True
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_SEQ_LEN,  # older TRL called this max_seq_length
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_train_ds,
    eval_dataset=sft_val_ds,
    peft_config=peft_config,
    processing_class=tokenizer,  # older TRL: tokenizer=tokenizer
)
sft_trainer.train()
sft_trainer.save_model(str(SFT_DIR))
print("Saved SFT adapter ->", SFT_DIR)

## 7. Build DPO preference pairs

Per proposal §2.1, `chosen` is the grounded/concise gold text and `rejected` is a degraded variant. These two strategies need **no teacher rollout** (fully programmatic, CPU-only):

- **dataset_description** → *verbosity* rejection: gold padded with generic rambling filler (trains against the verbosity penalty in §5).
- **column_description** → *cross-contamination* rejection: another column's gold description — plausible-sounding but ungrounded in this column's schema.

A baseline zero-shot verbose rollout from the base model could be added later as a third strategy.

In [ ]:
# @dryrun
FILLER = [
    "This information may be useful for a wide variety of analytical and reporting purposes across many different contexts.",
    "It is important to note that the underlying records are maintained over time and can be referenced for numerous downstream applications.",
    "Stakeholders and interested parties may find this resource valuable when conducting research, performing analysis, or preparing summaries.",
]


def make_verbose(text, k=3):
    return (text.rstrip() + " " + " ".join(FILLER[:k])).strip()


def build_dpo_pairs(ds_examples, col_examples, seed=42, max_pairs=None):
    rng = random.Random(seed)
    pairs = []
    for ex in ds_examples:  # verbosity preference
        pairs.append(
            {
                "task": ex["task"],
                "uid": ex["uid"],
                "prompt": ex["prompt_messages"],
                "chosen": [{"role": "assistant", "content": ex["target"]}],
                "rejected": [
                    {"role": "assistant", "content": make_verbose(ex["target"])}
                ],
            }
        )
    pool = [e["target"] for e in col_examples]
    for ex in col_examples:  # cross-contamination preference
        bad = ex["target"]
        for _ in range(8):
            cand = rng.choice(pool)
            if cand.strip() != ex["target"].strip():
                bad = cand
                break
        pairs.append(
            {
                "task": ex["task"],
                "uid": ex["uid"],
                "prompt": ex["prompt_messages"],
                "chosen": [{"role": "assistant", "content": ex["target"]}],
                "rejected": [{"role": "assistant", "content": bad}],
            }
        )
    rng.shuffle(pairs)
    return pairs[:max_pairs] if max_pairs else pairs


train_ds_ex, train_col_ex = build_examples_for_uids(records, splits["train"])
dpo_pairs = build_dpo_pairs(
    train_ds_ex, train_col_ex, seed=SEED, max_pairs=DPO_MAX_PAIRS
)


def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


write_jsonl(DATA_DIR / "dpo_pairs.jsonl", dpo_pairs)
print(f"{len(dpo_pairs)} preference pairs -> {DATA_DIR/'dpo_pairs.jsonl'}")
print("example rejected (verbose):\n", make_verbose(train_ds_ex[0]["target"])[:400])

## 8. DPO pass *(GPU)*

Continues training the SFT adapter. With a PEFT model and `ref_model=None`, TRL uses the adapter-disabled base as the reference policy.

In [ ]:
from peft import PeftModel
from trl import DPOConfig, DPOTrainer

# reload base in 4-bit, then load the SFT adapter as the (trainable) starting policy
dpo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
dpo_base = prepare_model_for_kbit_training(dpo_base)
dpo_base.config.use_cache = False
dpo_model = PeftModel.from_pretrained(dpo_base, str(SFT_DIR), is_trainable=True)

dpo_ds = Dataset.from_list(
    [
        {"prompt": p["prompt"], "chosen": p["chosen"], "rejected": p["rejected"]}
        for p in dpo_pairs
    ]
)

dpo_args = DPOConfig(
    output_dir=str(DPO_DIR),
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=DPO_LR,
    beta=DPO_BETA,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,  # on a T4 set bf16=False, fp16=True
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Recent TRL (>=0.20) collapsed DPO's max_prompt_length/max_completion_length into a
    # single max_length bounding the whole prompt+completion; truncation_mode controls the cut.
    max_length=MAX_SEQ_LEN,
    truncation_mode="keep_start",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
)
dpo_trainer.train()
dpo_trainer.save_model(str(DPO_DIR))
print("Saved final (SFT+DPO) adapter ->", DPO_DIR)

## 9. Quick inference sanity check *(GPU)*

Generate on a couple of **validation** datasets (test stays untouched for the eval notebook).

In [ ]:
import re

_THINK_RE = re.compile(
    r"^\s*<think>.*?</think>\s*", re.DOTALL
)  # defensive: drop any leading think block


def generate(prompt_messages, max_new_tokens=256):
    # enable_thinking=False makes the prompt end with the empty <think></think> block,
    # matching how training targets were rendered (see SFT cell).
    try:
        text = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:  # tokenizers without the enable_thinking kwarg
        text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
    inputs = tokenizer(text, return_tensors="pt").to(dpo_model.device)
    out = dpo_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    ).strip()
    return _THINK_RE.sub("", decoded).strip()


dpo_model.eval()

val_ds_ex, val_col_ex = build_examples_for_uids(records, splits["val"])
for ex, mnt in [(val_ds_ex[0], 256), (val_col_ex[0], 64)]:
    print(
        f"\n===== {ex['task']}  ({ex['uid']}{'/' + ex['column'] if ex['column'] else ''}) ====="
    )
    print("PRED:", generate(ex["prompt_messages"], mnt))
    print("GOLD:", ex["target"][:300])

## Outputs

| Artifact | Description |
| --- | --- |
| `splits.json` | held-out-by-dataset train/val/test UID lists (shared with eval) |
| `sft_data/sft_train.jsonl`, `sft_val.jsonl` | chat-formatted SFT examples (both tasks) |
| `sft_data/dpo_pairs.jsonl` | DPO preference pairs |
| `adapters/qwen3-8b-desc-sft/` | SFT adapter |
| `adapters/qwen3-8b-desc-dpo/` | **final** SFT+DPO adapter (loaded by `evaluate_descriptions.ipynb`) |